In [1]:
# Step 1 — pin numpy first to prevent binary incompatibility on Python 3.11
%pip install "numpy>=1.26,<2.0" --quiet

# Step 2 — install all other dependencies
%pip install langchain langchain-core langchain-community langchain-groq \
             langchain-openai langchain-anthropic langchain-text-splitters \
             pdfplumber pandas tabulate --quiet

print("\n✅ Installation complete.")
print("⚠️  Restart the kernel before running the next cell.")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.

✅ Installation complete.
⚠️  Restart the kernel before running the next cell.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import re
import json
import pdfplumber
from pathlib     import Path
from getpass     import getpass
from textwrap    import dedent

# LangChain components
from langchain_core.prompts        import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_anthropic           import ChatAnthropic
from langchain_groq                import ChatGroq
from langchain_openai              import ChatOpenAI

print("All imports successful.")

All imports successful.


In [3]:
# ── CHOOSE YOUR PROVIDER ─────────────────────────────────────────────────────
PROVIDER = "groq"   # "groq" | "claude" | "openai"
# ─────────────────────────────────────────────────────────────────────────────

if PROVIDER == "groq":
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")
    llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.2)
    print("LLM: Groq — llama-3.3-70b-versatile (free tier)")

elif PROVIDER == "claude":
    os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")
    llm = ChatAnthropic(model="claude-sonnet-4-6", temperature=0.2)
    print("LLM: Anthropic — claude-sonnet-4-6")

elif PROVIDER == "openai":
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
    print("LLM: OpenAI — gpt-4o-mini")

else:
    raise ValueError(f"Unknown PROVIDER: '{PROVIDER}'. Choose groq, claude, or openai.")

print("\nConfiguration complete.")

LLM: Groq — llama-3.3-70b-versatile (free tier)

Configuration complete.


In [4]:
SAMPLE_RESUME = dedent("""
    Ravi Kumar
    Bangalore, India  |  ravi.kumar@email.com  |  github.com/ravikumar

    SUMMARY
    Python developer with 3 years of experience in machine learning and data engineering.
    Worked on several ML projects and data pipelines. Interested in AI and NLP.

    SKILLS
    Python, Pandas, NumPy, Scikit-learn, TensorFlow, SQL, Git, Linux
    Basic knowledge of NLP and transformers

    EXPERIENCE

    ML Engineer — DataTech Solutions, Bangalore (2022–2024)
    - Worked on machine learning models for customer churn prediction
    - Helped build data pipelines using Python and SQL
    - Did some NLP work for text classification tasks
    - Wrote scripts to automate reporting tasks

    Junior Data Analyst — InfoSoft Pvt Ltd, Pune (2021–2022)
    - Analysed large datasets and created dashboards
    - Wrote SQL queries for business reporting
    - Assisted senior engineers with Python scripts

    EDUCATION
    B.Tech Computer Science — VIT University, 2021

    PROJECTS
    - Built a chatbot using Python (rule-based)
    - Sentiment analysis on Twitter data using NLTK
""").strip()

SAMPLE_JD = dedent("""
    Job Title: GenAI Engineer
    Company: TechCorp India, Bangalore

    About the Role:
    We are looking for a GenAI Engineer to build and deploy LLM-powered applications.
    You will work on RAG pipelines, prompt engineering, and production AI systems.

    Required Skills:
    - Strong Python programming (3+ years)
    - Experience with LLMs — OpenAI, Anthropic, or open-source models
    - Hands-on experience with LangChain or LlamaIndex
    - Knowledge of vector databases — FAISS, Chroma, or Pinecone
    - RAG (Retrieval Augmented Generation) pipeline development
    - Prompt engineering and fine-tuning experience
    - REST API development using FastAPI or Flask

    Good to Have:
    - Cloud deployment — AWS or GCP
    - Docker and containerisation
    - MLOps and model monitoring

    Responsibilities:
    - Build production RAG applications
    - Design and optimise LLM prompts
    - Integrate LLMs with internal tools and databases
    - Write clean, well-tested Python code
    - Deploy and monitor AI applications in production
""").strip()

print(f"Resume length  : {len(SAMPLE_RESUME.split())} words")
print(f"JD length      : {len(SAMPLE_JD.split())} words")
print("\nSample data ready.")

Resume length  : 147 words
JD length      : 145 words

Sample data ready.


In [5]:
def parse_resume(source: str) -> str:
    """
    Parse a resume from a PDF file path or a plain text string.
    Returns clean plain text suitable for LLM processing.
    """
    p = Path(source)

    # PDF path
    if p.suffix.lower() == '.pdf' and p.exists():
        pages = []
        with pdfplumber.open(p) as pdf:
            for page in pdf.pages:
                text = page.extract_text()
                if text and text.strip():
                    pages.append(text.strip())
        return '\n'.join(pages)

    # Plain text string — clean and return
    return re.sub(r'\n{3,}', '\n\n', source.strip())

# Test with sample resume
parsed = parse_resume(SAMPLE_RESUME)
print(f"Parsed resume ({len(parsed.split())} words):")
print("-" * 50)
print(parsed[:400])
print("...")

Parsed resume (147 words):
--------------------------------------------------
Ravi Kumar
Bangalore, India  |  ravi.kumar@email.com  |  github.com/ravikumar

SUMMARY
Python developer with 3 years of experience in machine learning and data engineering.
Worked on several ML projects and data pipelines. Interested in AI and NLP.

SKILLS
Python, Pandas, NumPy, Scikit-learn, TensorFlow, SQL, Git, Linux
Basic knowledge of NLP and transformers

EXPERIENCE

ML Engineer — DataTech So
...


In [6]:
SCORE_PROMPT_TEMPLATE = """\
You are an expert ATS (Applicant Tracking System) analyser.
Compare the resume against the job description and return a JSON analysis.

RULES:
1. Return ONLY valid JSON — no explanation, no markdown, no backticks.
2. Score should reflect genuine skill match (0=no match, 100=perfect match).
3. Only list missing skills that are GENUINELY absent from the resume.
   Do not hallucinate gaps. If a skill is present, do not list it as missing.
4. weak_bullets: list resume bullet points that are vague, generic, or lack metrics.

REQUIRED JSON SCHEMA:
{{
  \"score\"        : <integer 0-100>,
  \"matched\"      : [<skills present in both resume and JD>],
  \"missing\"      : [<skills in JD but not in resume>],
  \"weak_bullets\" : [<exact bullet text that is vague or lacks impact>]
}}

RESUME:
──────────────────────────────────────
{resume}
──────────────────────────────────────

JOB DESCRIPTION:
──────────────────────────────────────
{job_description}
──────────────────────────────────────

JSON Analysis:"""

score_prompt = PromptTemplate(
    template        = SCORE_PROMPT_TEMPLATE,
    input_variables = ['resume', 'job_description'],
)

score_chain = score_prompt | llm | StrOutputParser()

def parse_json_response(raw: str) -> dict:
    """
    Parse LLM JSON response — strips markdown fences if present.
    """
    # Remove markdown code fences the LLM sometimes adds
    cleaned = re.sub(r'```(?:json)?\s*', '', raw, flags=re.IGNORECASE).strip()
    cleaned = cleaned.strip('`').strip()
    return json.loads(cleaned)

print("Scoring chain ready.")

Scoring chain ready.


In [7]:
resume_text = parse_resume(SAMPLE_RESUME)

raw_score = score_chain.invoke({
    'resume'          : resume_text,
    'job_description' : SAMPLE_JD,
})

analysis = parse_json_response(raw_score)

print("=" * 60)
print("ATS ANALYSIS RESULT")
print("=" * 60)
print(f"\nMatch Score : {analysis['score']}/100")

print(f"\nMatched Skills ({len(analysis['matched'])}):")
for skill in analysis['matched']:
    print(f"  ✅ {skill}")

print(f"\nMissing Skills ({len(analysis['missing'])}):")
for skill in analysis['missing']:
    print(f"  ❌ {skill}")

print(f"\nWeak Bullets ({len(analysis['weak_bullets'])}):")
for i, bullet in enumerate(analysis['weak_bullets'], 1):
    print(f"  {i}. {bullet[:80]}...")

ATS ANALYSIS RESULT

Match Score : 20/100

Matched Skills (2):
  ✅ Python
  ✅ SQL

Missing Skills (6):
  ❌ LLMs — OpenAI, Anthropic, or open-source models
  ❌ LangChain or LlamaIndex
  ❌ vector databases — FAISS, Chroma, or Pinecone
  ❌ RAG pipeline development
  ❌ prompt engineering
  ❌ REST API development using FastAPI or Flask

Weak Bullets (5):
  1. Worked on machine learning models for customer churn prediction...
  2. Helped build data pipelines using Python and SQL...
  3. Did some NLP work for text classification tasks...
  4. Wrote scripts to automate reporting tasks...
  5. Analysed large datasets and created dashboards...


In [8]:
REWRITE_PROMPT_TEMPLATE = """\
You are an expert resume writer helping a candidate pass ATS filters.
Rewrite the given resume bullet point to be stronger and more ATS-friendly.

RULES:
1. Base the rewrite on the ORIGINAL bullet — do not invent experience.
2. Use specific keywords and language from the JOB DESCRIPTION.
3. Add metrics or concrete outcomes if they can be reasonably inferred.
4. Return ONLY the rewritten bullet — no explanation, no numbering.
5. Keep it to one line, starting with a strong action verb.

ORIGINAL BULLET:
{original_bullet}

JOB DESCRIPTION KEYWORDS TO INCORPORATE:
{job_description}

REWRITTEN BULLET:"""

rewrite_prompt = PromptTemplate(
    template        = REWRITE_PROMPT_TEMPLATE,
    input_variables = ['original_bullet', 'job_description'],
)

rewrite_chain = rewrite_prompt | llm | StrOutputParser()

def rewrite_bullets(weak_bullets: list, job_description: str) -> list:
    """
    Rewrite each weak bullet using JD keywords.
    Returns list of dicts: {original, rewritten}
    """
    results = []
    for bullet in weak_bullets:
        rewritten = rewrite_chain.invoke({
            'original_bullet' : bullet,
            'job_description' : job_description,
        }).strip()
        results.append({
            'original' : bullet,
            'rewritten': rewritten,
        })
    return results

print("Rewriter chain ready.")

Rewriter chain ready.


In [9]:
rewrites = rewrite_bullets(analysis['weak_bullets'], SAMPLE_JD)

print("=" * 65)
print("BULLET REWRITES")
print("=" * 65)

for i, item in enumerate(rewrites, 1):
    print(f"\n[{i}] ORIGINAL:")
    print(f"    {item['original']}")
    print(f"\n    REWRITTEN:")
    print(f"    {item['rewritten']}")
    print("-" * 65)

BULLET REWRITES

[1] ORIGINAL:
    Worked on machine learning models for customer churn prediction

    REWRITTEN:
    Developed and deployed machine learning models utilizing LLMs and RAG pipelines to predict customer churn, leveraging Python programming and vector databases to drive predictive insights and inform business decisions.
-----------------------------------------------------------------

[2] ORIGINAL:
    Helped build data pipelines using Python and SQL

    REWRITTEN:
    Developed scalable data pipelines leveraging Python and SQL, laying the groundwork for large language model (LLM) integration and potential RAG pipeline development, with a focus on building a foundation for production-ready AI applications.
-----------------------------------------------------------------

[3] ORIGINAL:
    Did some NLP work for text classification tasks

    REWRITTEN:
    Developed and deployed natural language processing solutions for text classification tasks utilizing large languag

In [10]:
class ATSScorer:
    """
    Production-ready ATS Resume Scorer and Bullet Rewriter.
    Works with any resume (PDF or text) and any job description.
    """

    _SCORE_PROMPT = """\
You are an expert ATS analyser. Compare the resume against the job description.
Return ONLY valid JSON — no explanation, no markdown, no backticks.
Only list skills as missing if they are GENUINELY absent from the resume.

JSON SCHEMA:
{{
  \"score\"        : <integer 0-100>,
  \"matched\"      : [<skills present in both>],
  \"missing\"      : [<skills in JD but not in resume>],
  \"weak_bullets\" : [<vague or low-impact bullet text>]
}}

RESUME:\n{resume}\n\nJOB DESCRIPTION:\n{job_description}\n\nJSON:"""

    _REWRITE_PROMPT = """\
Rewrite this resume bullet to be ATS-friendly using keywords from the JD.
Rules: base on original only, use JD keywords, add metrics if inferable,
return ONLY the rewritten bullet, start with an action verb.

ORIGINAL: {original_bullet}\nJD: {job_description}\nREWRITTEN:"""

    def __init__(self, provider: str = 'groq'):
        self.provider = provider
        self._init_llm()
        self._init_chains()

    def _init_llm(self):
        """Initialise LLM based on provider string."""
        if self.provider == 'groq':
            self.llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0.2)
        elif self.provider == 'claude':
            self.llm = ChatAnthropic(model='claude-sonnet-4-6', temperature=0.2)
        elif self.provider == 'openai':
            self.llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.2)
        else:
            raise ValueError(f"Unknown provider: '{self.provider}'")

    def _init_chains(self):
        """Build scoring and rewriting chains."""
        sp = PromptTemplate(
            template=self._SCORE_PROMPT,
            input_variables=['resume', 'job_description']
        )
        rp = PromptTemplate(
            template=self._REWRITE_PROMPT,
            input_variables=['original_bullet', 'job_description']
        )
        self._score_chain   = sp | self.llm | StrOutputParser()
        self._rewrite_chain = rp | self.llm | StrOutputParser()

    @staticmethod
    def _grade(score: int) -> str:
        """Convert numeric score to letter grade."""
        if score >= 80: return 'A — Strong match'
        if score >= 60: return 'B — Good match, minor gaps'
        if score >= 40: return 'C — Moderate match, needs work'
        return 'D — Weak match, significant gaps'

    def score(self, resume: str, job_description: str) -> dict:
        """
        Score a resume against a job description.
        Returns complete analysis with rewrites.
        """
        # Stage 1 — Parse
        resume_text = parse_resume(resume)

        # Stage 2 — Score and gap analysis
        raw = self._score_chain.invoke({
            'resume'          : resume_text,
            'job_description' : job_description,
        })
        analysis = parse_json_response(raw)

        # Stage 3 — Rewrite weak bullets
        rewrites = []
        for bullet in analysis.get('weak_bullets', []):
            rewritten = self._rewrite_chain.invoke({
                'original_bullet' : bullet,
                'job_description' : job_description,
            }).strip()
            rewrites.append({'original': bullet, 'rewritten': rewritten})

        return {
            'score'        : analysis.get('score', 0),
            'grade'        : self._grade(analysis.get('score', 0)),
            'matched'      : analysis.get('matched', []),
            'missing'      : analysis.get('missing', []),
            'weak_bullets' : analysis.get('weak_bullets', []),
            'rewrites'     : rewrites,
        }

print("ATSScorer class defined.")

ATSScorer class defined.


In [11]:
scorer = ATSScorer(provider=PROVIDER)

report = scorer.score(
    resume          = SAMPLE_RESUME,
    job_description = SAMPLE_JD,
)

print("=" * 65)
print("COMPLETE ATS REPORT — ATSScorer class")
print("=" * 65)

print(f"\nScore : {report['score']}/100")
print(f"Grade : {report['grade']}")

print(f"\nMatched Skills ({len(report['matched'])}):")
for s in report['matched']:
    print(f"  ✅  {s}")

print(f"\nMissing Skills ({len(report['missing'])}):")
for s in report['missing']:
    print(f"  ❌  {s}")

print(f"\nBullet Rewrites ({len(report['rewrites'])}):")
for i, r in enumerate(report['rewrites'], 1):
    print(f"\n  [{i}] Before: {r['original'][:70]}...")
    print(f"       After : {r['rewritten']}")
print("\n" + "=" * 65)

COMPLETE ATS REPORT — ATSScorer class

Score : 20/100
Grade : D — Weak match, significant gaps

Matched Skills (2):
  ✅  Python
  ✅  SQL

Missing Skills (6):
  ❌  LLMs — OpenAI, Anthropic, or open-source models
  ❌  LangChain or LlamaIndex
  ❌  vector databases — FAISS, Chroma, or Pinecone
  ❌  RAG pipeline development
  ❌  prompt engineering
  ❌  REST API development using FastAPI or Flask

Bullet Rewrites (3):

  [1] Before: Did some NLP work for text classification tasks...
       After : Developed natural language processing solutions for text classification tasks utilizing large language models, with a focus on prompt engineering and fine-tuning, resulting in improved model accuracy and efficiency, and laying the groundwork for potential integration with RAG pipelines and vector databases.

  [2] Before: Assisted senior engineers with Python scripts...
       After : Developed Python scripts utilizing LLMs and RAG pipelines, assisting senior engineers in building and deploying AI 